# MiWay GTFS Route Efficiency Project - Notebook 2

## Route geometry, directness, stops, and stop density

This notebook builds the first real efficiency metrics for MiWay routes using the GTFS scheduled feed.

It calculates, for routes operating on **Tuesday, May 5, 2026**:

1. route path length from `shapes.txt`,
2. start-to-end straight-line distance,
3. directness ratio,
4. unique stop count,
5. stop density,
6. and a clean route-level table for later scoring.

The output of this notebook becomes the geometry foundation for the route efficiency ranking.

## 1. Import libraries

We use common Python packages only. The distance calculations use a small Haversine helper function, which estimates distance over the Earth's surface from latitude and longitude.

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 140)

## 2. Locate project folders and the GTFS ZIP

This block works whether Jupyter starts in the project root or inside the `Notebooks/` folder.

In [ ]:
candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'Data' / 'google_transit.zip').exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find Data/google_transit.zip from this notebook location.')

DATA_DIR = PROJECT_ROOT / 'Data'
OUTPUT_DIR = PROJECT_ROOT / 'Outputs'
ZIP_PATH = DATA_DIR / 'google_transit.zip'

OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT.resolve()}')
print(f'GTFS ZIP:     {ZIP_PATH.resolve()}')
print(f'Outputs:      {OUTPUT_DIR.resolve()}')

## 3. Load the GTFS tables

Notebook 2 reloads the source GTFS files directly so it can be run independently from Notebook 1.

In [ ]:
def read_gtfs_table(zip_path: Path, filename: str) -> pd.DataFrame:
    """Read one GTFS text table directly from the ZIP archive."""
    with zipfile.ZipFile(zip_path, 'r') as zf:
        if filename not in zf.namelist():
            raise FileNotFoundError(f'{filename} is missing from {zip_path.name}')
        with zf.open(filename) as file:
            return pd.read_csv(file)


routes = read_gtfs_table(ZIP_PATH, 'routes.txt')
trips = read_gtfs_table(ZIP_PATH, 'trips.txt')
stop_times = read_gtfs_table(ZIP_PATH, 'stop_times.txt')
stops = read_gtfs_table(ZIP_PATH, 'stops.txt')
shapes = read_gtfs_table(ZIP_PATH, 'shapes.txt')
calendar_dates = read_gtfs_table(ZIP_PATH, 'calendar_dates.txt')
feed_info = read_gtfs_table(ZIP_PATH, 'feed_info.txt')

print('Loaded GTFS tables successfully.')

## 4. Select the same representative weekday

We keep using **Tuesday, May 5, 2026** so these metrics line up with Notebook 1.

In [ ]:
ANALYSIS_DATE = pd.Timestamp('2026-05-05')
ANALYSIS_DATE_INT = int(ANALYSIS_DATE.strftime('%Y%m%d'))

feed_start = pd.to_datetime(str(feed_info.loc[0, 'feed_start_date']), format='%Y%m%d')
feed_end = pd.to_datetime(str(feed_info.loc[0, 'feed_end_date']), format='%Y%m%d')

if not (feed_start <= ANALYSIS_DATE <= feed_end):
    raise ValueError('Analysis date is outside the feed validity period.')

active_services = calendar_dates.loc[
    (calendar_dates['date'] == ANALYSIS_DATE_INT) &
    (calendar_dates['exception_type'] == 1),
    'service_id'
].drop_duplicates()

weekday_trips = trips.loc[trips['service_id'].isin(active_services)].copy()

weekday_trips_with_routes = weekday_trips.merge(
    routes,
    on='route_id',
    how='left',
    validate='many_to_one'
)

print(f'Feed period: {feed_start:%B %d, %Y} to {feed_end:%B %d, %Y}')
print(f'Analysis date: {ANALYSIS_DATE:%A, %B %d, %Y}')
print(f'Active services: {len(active_services)}')
print(f'Active weekday trips: {len(weekday_trips):,}')
print(f'Active routes: {weekday_trips["route_id"].nunique()}')

## 5. Calculate shape-level path length

Each `shape_id` is an ordered list of latitude/longitude points describing the path a bus follows. We calculate the distance between each consecutive pair of points, then sum those segment distances for each shape.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized Haversine distance in kilometres."""
    earth_radius_km = 6371.0088

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return earth_radius_km * c


shapes_sorted = shapes.sort_values(['shape_id', 'shape_pt_sequence']).copy()

shapes_sorted['next_lat'] = shapes_sorted.groupby('shape_id')['shape_pt_lat'].shift(-1)
shapes_sorted['next_lon'] = shapes_sorted.groupby('shape_id')['shape_pt_lon'].shift(-1)

shapes_sorted['segment_km'] = haversine_km(
    shapes_sorted['shape_pt_lat'],
    shapes_sorted['shape_pt_lon'],
    shapes_sorted['next_lat'],
    shapes_sorted['next_lon']
)

shapes_sorted['segment_km'] = shapes_sorted['segment_km'].fillna(0)

shape_lengths = (
    shapes_sorted
    .groupby('shape_id', as_index=False)
    .agg(
        shape_path_km=('segment_km', 'sum'),
        shape_points=('shape_pt_sequence', 'count'),
        start_lat=('shape_pt_lat', 'first'),
        start_lon=('shape_pt_lon', 'first'),
        end_lat=('shape_pt_lat', 'last'),
        end_lon=('shape_pt_lon', 'last')
    )
)

shape_lengths['straight_line_km'] = haversine_km(
    shape_lengths['start_lat'],
    shape_lengths['start_lon'],
    shape_lengths['end_lat'],
    shape_lengths['end_lon']
)

shape_lengths['directness_ratio'] = np.where(
    shape_lengths['shape_path_km'] > 0,
    shape_lengths['straight_line_km'] / shape_lengths['shape_path_km'],
    np.nan
)

shape_lengths.head()

## 6. Keep only shapes used on the analysis date

A route can have multiple shapes because it may have branches, short turns, or direction variants. We count how often each shape appears in the selected weekday schedule.

In [ ]:
weekday_shape_usage = (
    weekday_trips_with_routes
    .groupby([
        'route_id', 'route_short_name', 'route_long_name',
        'direction_id', 'shape_id'
    ], as_index=False)
    .agg(scheduled_trips=('trip_id', 'nunique'))
)

weekday_shape_metrics = weekday_shape_usage.merge(
    shape_lengths,
    on='shape_id',
    how='left',
    validate='many_to_one'
)

weekday_shape_metrics = weekday_shape_metrics.sort_values(
    ['route_short_name', 'direction_id', 'scheduled_trips'],
    ascending=[True, True, False]
)

weekday_shape_metrics.head(20)

## 7. Summarize geometry by route

For route-level metrics, we use scheduled-trip-weighted averages. This means a shape used by 80 trips influences the route average more than a rare branch used by 2 trips.

In [ ]:
def weighted_average(group: pd.DataFrame, value_col: str, weight_col: str = 'scheduled_trips') -> float:
    valid = group[[value_col, weight_col]].dropna()
    if valid.empty or valid[weight_col].sum() == 0:
        return np.nan
    return np.average(valid[value_col], weights=valid[weight_col])


route_geometry_summary = (
    weekday_shape_metrics
    .groupby(['route_id', 'route_short_name', 'route_long_name'])
    .apply(lambda g: pd.Series({
        'scheduled_trips': g['scheduled_trips'].sum(),
        'directions': g['direction_id'].nunique(),
        'unique_shapes': g['shape_id'].nunique(),
        'avg_shape_path_km': weighted_average(g, 'shape_path_km'),
        'median_shape_path_km': g['shape_path_km'].median(),
        'max_shape_path_km': g['shape_path_km'].max(),
        'avg_straight_line_km': weighted_average(g, 'straight_line_km'),
        'avg_directness_ratio': weighted_average(g, 'directness_ratio'),
    }))
    .reset_index()
)

route_geometry_summary = route_geometry_summary.sort_values(
    ['avg_directness_ratio', 'scheduled_trips'],
    ascending=[True, False]
)

route_geometry_summary.head(20)

## 8. Calculate route stop counts

To estimate how stop-heavy a route is, we count the unique stops served by all trips on the analysis date.

In [ ]:
weekday_trip_ids = weekday_trips_with_routes[['trip_id', 'route_id']].copy()

weekday_stop_times = stop_times.merge(
    weekday_trip_ids,
    on='trip_id',
    how='inner',
    validate='many_to_one'
)

trip_stop_counts = (
    weekday_stop_times
    .groupby(['route_id', 'trip_id'], as_index=False)
    .agg(stops_on_trip=('stop_id', 'count'))
)

route_stop_summary = (
    weekday_stop_times
    .groupby('route_id', as_index=False)
    .agg(
        unique_stops=('stop_id', 'nunique'),
        total_stop_visits=('stop_id', 'count')
    )
    .merge(
        trip_stop_counts.groupby('route_id', as_index=False).agg(
            avg_stops_per_trip=('stops_on_trip', 'mean')
        ),
        on='route_id',
        how='left',
        validate='one_to_one'
    )
)

route_stop_summary.head()

## 9. Combine geometry and stop metrics

Stop density is calculated as unique stops per kilometre of average route path. Higher stop density can mean better access, but it can also slow down travel speeds.

In [ ]:
route_geometry_stop_metrics = route_geometry_summary.merge(
    route_stop_summary,
    on='route_id',
    how='left',
    validate='one_to_one'
)

route_geometry_stop_metrics['stops_per_km'] = (
    route_geometry_stop_metrics['unique_stops'] /
    route_geometry_stop_metrics['avg_shape_path_km']
)

numeric_cols = [
    'avg_shape_path_km', 'median_shape_path_km', 'max_shape_path_km',
    'avg_straight_line_km', 'avg_directness_ratio', 'avg_stops_per_trip',
    'stops_per_km'
]
route_geometry_stop_metrics[numeric_cols] = route_geometry_stop_metrics[numeric_cols].round(3)

route_geometry_stop_metrics = route_geometry_stop_metrics.sort_values(
    ['avg_directness_ratio', 'stops_per_km'],
    ascending=[True, False]
)

route_geometry_stop_metrics.head(25)

## 10. Identify routes with lower directness

A lower directness ratio means the scheduled path is much longer than the straight-line distance between its endpoints. This is not automatically bad, because transit routes often bend to serve important destinations, but it is a useful efficiency signal.

In [ ]:
least_direct_routes = route_geometry_stop_metrics[
    ['route_short_name', 'route_long_name', 'scheduled_trips', 'avg_shape_path_km',
     'avg_straight_line_km', 'avg_directness_ratio', 'unique_stops', 'stops_per_km']
].head(15)

least_direct_routes

## 11. Identify routes with high stop density

Routes with many stops per kilometre may provide fine-grained local access but may also have slower operating speeds. This becomes more meaningful once Notebook 3 adds scheduled travel speed.

In [ ]:
highest_stop_density_routes = route_geometry_stop_metrics.sort_values(
    'stops_per_km',
    ascending=False
)[
    ['route_short_name', 'route_long_name', 'scheduled_trips', 'avg_shape_path_km',
     'unique_stops', 'stops_per_km', 'avg_directness_ratio']
].head(15)

highest_stop_density_routes

## 12. Sanity checks

These checks help confirm the output is complete enough to use in the next notebook.

In [ ]:
sanity_checks = {
    'active weekday routes': weekday_trips_with_routes['route_id'].nunique(),
    'routes in final metric table': len(route_geometry_stop_metrics),
    'shapes used on analysis date': weekday_shape_metrics['shape_id'].nunique(),
    'missing shape lengths': weekday_shape_metrics['shape_path_km'].isna().sum(),
    'routes missing unique stop count': route_geometry_stop_metrics['unique_stops'].isna().sum(),
    'routes missing directness ratio': route_geometry_stop_metrics['avg_directness_ratio'].isna().sum(),
}

pd.Series(sanity_checks, name='value').to_frame()

## 13. Export Notebook 2 outputs

The final CSV is designed to be used by later notebooks for speed, frequency, and final efficiency scoring.

In [ ]:
shape_metrics_path = OUTPUT_DIR / 'weekday_shape_metrics_2026-05-05.csv'
route_metrics_path = OUTPUT_DIR / 'route_geometry_stop_metrics_2026-05-05.csv'

weekday_shape_metrics.to_csv(shape_metrics_path, index=False)
route_geometry_stop_metrics.to_csv(route_metrics_path, index=False)

print('Exported:')
print('-', shape_metrics_path)
print('-', route_metrics_path)

# What this notebook accomplished

Notebook 2 created the first route efficiency metrics:

- average scheduled path length,
- average straight-line endpoint distance,
- directness ratio,
- unique stop count,
- average stops per trip,
- and stops per kilometre.

## Next notebook

Notebook 3 should calculate **scheduled service performance metrics**:

1. trip duration from first and last stop times,
2. scheduled average speed,
3. daily trip frequency by route,
4. approximate average headway,
5. peak-period versus off-peak service levels.